In [1]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="0,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
import time

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

/home/dataconv/anaconda3/envs/shyoon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random

def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED']=str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic=True    
    torch.backends.cudnn.benchmark=True
    
seed_everything(42)

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,c2687961-0957-45cb-bae0-42314e38f790,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,26830122-8240-40a9-aaff-d9731d53b197,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,268116a9-5ecb-4364-8da4-4a648f9d5b43,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,efb4810e-637b-4954-a776-3c2d05d1290c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,99817eba-d32a-4c4d-9fe2-93a50ae1d367,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [5]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [6]:
# import pandas as pd
# evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'

# evidence_text = pd.read_csv(evidence_test_path)

In [7]:
# evidence_text_list = evidence_text['text'].tolist()

In [8]:
# from langchain_text_splitters import TokenTextSplitter

# text_splitter = TokenTextSplitter(
#     chunk_size=500,  # 청크 크기를 10으로 설정합니다.
#     chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
# )
# # combined_text = " ".join(evidence_text_list)
# # texts = text_splitter.split_text(combined_text)
# split_texts = [text_splitter.split_text(text)[0] for text in evidence_text_list]
# print(split_texts[0])

In [9]:
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.vectorstores import FAISS

# # bm25 retriever와 faiss retriever를 초기화합니다.
# bm25_retriever = BM25Retriever.from_texts(
#     evidence_text_list,
# )
# bm25_retriever.k = 10  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

# embedding = model
# faiss_vectorstore = FAISS.from_texts(
#     evidence_text,
#     embedding,
# )
# faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# # 앙상블 retriever를 초기화합니다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.7, 0.3],
# )

In [10]:
# from langchain_community.document_transformers import LongContextReorder

# def bm25_retrieve(query):
#     bm25_result = bm25_retriever.invoke(query)
#     bm25_docs=list()

#     print("[BM25 Retriever]")
#     for doc in bm25_result:
#         # print(f"Content: {doc.page_content}")
#         # print()
#         bm25_docs.append(doc.page_content)
#     reordering = LongContextReorder()
#     bm25_docs = reordering.transform_documents(bm25_docs)
#     return bm25_docs

In [11]:
# res=bm25_retrieve("Who has the highest goals in world football?")
# res

In [15]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [8]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [9]:
def various_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [10]:
def various_answer(query, docs, first_ans=None):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}{2}
    Answer:
    """.format('\n'.join(docs), query, f"Prior Answer: {first_ans}")
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [11]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

# Baseline

In [12]:
from tqdm import tqdm
from evaluation import evaluate

seed_everything(24)

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    ans=answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/shyoon/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Final ans: ['Based on the provided information, the top goalscorers in international football are:1. Ali Daei - 109 goals2. Cristiano Ronaldo - 99 goals3. Ferenc Puskás - 84 goals4. Kunishige Kamamoto - 80 goals5. Godfrey Chitalu - 79 goals6. Hussein Saeed - 78 goals7. Zainal Abidin - 78 goals8. Pelé - 77 goals9. Bashar Abdullah - 75 goals10. Sunil Chhetri - 72 goalsHowever, the player with the highest goals in world football is not explicitly stated in the provided information.']
Based on the provided information, the top goalscorers in international football are:1. Ali Daei - 109 goals2. Cristiano Ronaldo - 99 goals3. Ferenc Puskás - 84 goals4. Kunishige Kamamoto - 80 goals5. Godfrey Chitalu - 79 goals6. Hussein Saeed - 78 goals7. Zainal Abidin - 78 goals8. Pelé - 77 goals9. Bashar Abdullah - 75 goals10. Sunil Chhetri - 72 goalsHowever, the player with the highest goals in world football is not explicitly stated in the provided information.
Who has the highest goals in world football

  5%|▌         | 1/20 [00:38<12:12, 38.57s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.323673278093338, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.01449085958302021, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.01846776157617569, 'start': 88, 'end': 96, 'answer': 'Ali Daei'}
{'rougeLsum': 24.277456647398843, 'length': 78.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Final ans: ['The original artist of "The Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel.']
The original artist of "The Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel.
Who is the or

 10%|█         | 2/20 [00:49<06:40, 22.24s/it]

...

 95%|█████████▌| 19/20 [04:13<00:13, 13.05s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9788501262664795, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8789895176887512, 'start': 153, 'end': 163, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9475837349891663, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.8670162558555603, 'start': 153, 'end': 163, 'answer': 'Ethan Peck'}
{'rougeLsum': 71.42857142857143, 'length': 29.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Final ans: ['Yes, Microsoft Live Mo

100%|██████████| 20/20 [04:24<00:00, 13.22s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.21996232867240906, 'start': 54, 'end': 62, 'answer': 'software'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.285571813583374, 'start': 144, 'end': 157, 'answer': 'free download'}
{'rougeLsum': 38.095238095238095, 'length': 28.0, 'str_em': 0.0, 'Disambig-F1': 0.0}


rougeLsum      35.548461
length         25.150000
str_em         47.500000
Disambig-F1    47.343254
dtype: float64

# Answer RAG

In [13]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    first_ans=various_answer(query,retrieved_docs)
    print('First ans:', first_ans[0])
    ans_docs=retrieve_documents(first_ans[0])
    final_ans=various_answer(query,ans_docs,first_ans[0])
    print('Second ans:', final_ans[0])
    scores=evaluate(final_ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

First ans: Based on the provided context, the player with the highest goals in world football is Josef Bican with 805 goals.However, if we consider the number of goals scored in international football, the player with the highest goals is Ali Daei of Iran, who has scored 109 goals in 149 international appearances.Additionally, if we look at the number of goals scored in a single season, the player with the highest goals is Lionel Messi, who scored 73 goals in the 2011-12 season.It's worth noting that these records may change over time as new data becomes available, and different sources may have different rankings and records.
Second ans: Based on the provided context information, the player with the highest goals in world football is Josef Bican with 805 goals. However, if we consider the number of goals scored in international football, the player with the highest goals is Ali Daei of Iran, who has scored 109 goals in 149 international appearances.**Golden Short Answer 1:** Josef Bic

  5%|▌         | 1/20 [01:40<31:47, 100.41s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.6773802638053894, 'start': 98, 'end': 109, 'answer': 'Josef Bican'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.02799687162041664, 'start': 98, 'end': 109, 'answer': 'Josef Bican'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.00019910109404008836, 'start': 98, 'end': 109, 'answer': 'Josef Bican'}
{'rougeLsum': 35.932203389830505, 'length': 185.0, 'str_em': 66.66666666666666, 'Disambig-F1': 33.33333333333333}
First ans: Context information is below.    ---------------------    Document: The Sound of SilenceTitle: Homeward BoundDescription: SinglebySimon & GarfunkelFields:B-side: "We've Got a Groovy Thing Goin'"Released: September 12, 1965(1965-09-12)Recorded: June 15, 1965 (overd

 10%|█         | 2/20 [02:03<16:30, 55.01s/it] 

...

 95%|█████████▌| 19/20 [18:58<00:47, 47.97s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.969932496547699, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8799009919166565, 'start': 173, 'end': 183, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.8959749937057495, 'start': 75, 'end': 87, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.9614259600639343, 'start': 173, 'end': 183, 'answer': 'Ethan Peck'}
{'rougeLsum': 55.91397849462365, 'length': 50.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
First ans: Based on the provided con

100%|██████████| 20/20 [19:38<00:00, 58.91s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.767994225025177, 'start': 89, 'end': 97, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.14809392392635345, 'start': 89, 'end': 97, 'answer': 'freeware'}
{'rougeLsum': 38.46153846153847, 'length': 43.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum        30.086291
length         1628.950000
str_em           65.000000
Disambig-F1      45.369048
dtype: float64

In [22]:
scores_df.to_csv('./results/self-refine-02-07_results.csv', index=False)

In [23]:
import pandas as pd
import math
sf = pd.read_csv('results/self-refine-02-07_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

15
rougeLsum       35.494841
length         113.333333
str_em          48.888889
Disambig-F1     42.314815
dtype: float64
38.75509785353146


# Cheatseat RAG

In [9]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [26]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Your work is making cheat seat for answering follow question.
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Create a cheat seat to help other LLMs address relevant questions.
    Please do not write introduction message like 'Based on the provided information'
    Query: {1}
    Cheat seat:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [20]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [28]:
from tqdm import tqdm
from evaluation import evaluate

seed_everything(24)

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    ans=summarize(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

Final ans: ['**Top Scorers in World Football**1. **Ali Daei** (Iran) - 109 goals2. **Cristiano Ronaldo** (Portugal) - 99 goals3. **Ferenc Puskás** (Hungary) - 84 goals4. **Kunishige Kamamoto** (Japan) - 80 goals5. **Godfrey Chitalu** (Zambia) - 79 goals**Active Players with 500+ Goals**1. **Cristiano Ronaldo** (Portugal) - 738 goals2. **Lionel Messi** (Argentina) - 719 goals3. **Zlatan Ibrahimović** (Sweden) - 545 goals4. **Luis Suárez** (Uruguay) - 475 goals5. **Robert Lewandowski** (Poland) - 465 goals**Top Goalscorers in a Single Tournament**1. **Just Fontaine** (France, 1958) - 13 goals2. **Gerd Müller** (West Germany, 1970) - 10 goals3. **Pelé** (Brazil, 1958-1970) - 12 goals4. **Miroslav Klose** (Germany, 2002-2014) - 16 goals**Most Consecutive Hat-Tricks**1. **Sándor Kocsis** (Hungary, 1954) - 2 hat-tricks2. **Gerd Müller** (West Germany, 1970) - 2 hat-tricks**Most Goals Scored by a Substitute in a Match**1. **László Kiss** (Hungary) - 3 goals**Most Tournaments with at Least One

  5%|▌         | 1/20 [01:37<31:00, 97.93s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.13253313302993774, 'start': 38, 'end': 46, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.17154687643051147, 'start': 681, 'end': 695, 'answer': 'Miroslav Klose'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.08554711192846298, 'start': 38, 'end': 46, 'answer': 'Ali Daei'}
{'rougeLsum': 14.545454545454545, 'length': 167.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Final ans: ['**Original Artist:**Simon & Garfunkel**Context:** The song "The Sound of Silence" was written by Paul Simon and is a part of the duo\'s debut album "Wednesday Morning, 3 A.M." and their second studio album "Sounds of Silence". **Entities:**- Simon & Garfunkel- Paul Si

 10%|█         | 2/20 [02:18<19:20, 64.48s/it]

...

 95%|█████████▌| 19/20 [15:14<00:45, 45.99s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.8038073778152466, 'start': 32, 'end': 44, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.2914069890975952, 'start': 105, 'end': 115, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.07536713033914566, 'start': 32, 'end': 44, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.030207455158233643, 'start': 105, 'end': 115, 'answer': 'Ethan Peck'}
{'rougeLsum': 42.96296296296297, 'length': 86.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Final ans: ['*   **Entity:** Mic

100%|██████████| 20/20 [16:35<00:00, 49.78s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.09208977222442627, 'start': 1294, 'end': 1360, 'answer': 'video editing software that allows users to create and edit videos'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.11688575893640518, 'start': 1791, 'end': 1857, 'answer': 'video editing software that allows users to create and edit videos'}
{'rougeLsum': 10.149253731343283, 'length': 295.0, 'str_em': 50.0, 'Disambig-F1': 21.428571428571427}


rougeLsum       23.606964
length         139.900000
str_em          56.666667
Disambig-F1     48.724206
dtype: float64

In [29]:
scores_df.to_csv('./results/cheat-refine-02-08_results.csv', index=False)

# INFO RAG

In [31]:
def summarize(query, docs):
    prompt = """
    
Your task is to create a **detailed and structured cheat sheet** to provide comprehensive information for answering the query.  
The cheat sheet should include **as much relevant information as possible** extracted from the retrieved documents.  

**Instructions:**  
1️⃣ Extract **all relevant facts, statistics, and details** from the retrieved documents.  
2️⃣ **Prioritize the most important information at the top**, but **do not omit any useful details**.  
3️⃣ Ensure the cheat sheet contains **at least five structured entries**, but more is better.  
4️⃣ If retrieved documents contain multiple perspectives or conflicting information, **include all viewpoints**.  
5️⃣ Do not summarize excessively; **preserve specific details, names, numbers, and context**.

**Example Output Format:**  
("Most important fact 1.",  
 "Most important fact 2.",
"Most important fact 3.", 
"Most important fact 4.",   
 "Additional key fact 1.",  
 "Additional key fact 2.",  
 "Supporting information 1.",  
 "Supporting information 2.",  
 "Contextual background 1.",  
 "Contextual background 2.")

**Retrieved Documents:**  
---------------------  
{0}  
---------------------  

**Query:**  
{1}  

**Cheat Sheet:**  


    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [18]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [32]:
from tqdm import tqdm
from evaluation import evaluate

seed_everything(24)

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    summary=summarize(query,retrieved_docs)
    ans=answer(summary,query)
    print('Final ans:', summary)
    scores=evaluate(summary, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

Final ans: ['("Josef Bican",   "Josef Bican\'s record of 805 goals",   "Josef Bican\'s goals per match ratio",   "Josef Bican\'s career span",   "Josef Bican\'s active status as of 30 January 2020",   "Josef Bican\'s achievements and records in world football",   "Josef Bican\'s standing among other footballers with 500 or more goals",   "Josef Bican\'s legacy and impact on world football")Most important fact 1: Josef Bican holds the record for the highest number of goals scored in world football with 805 goals.Most important fact 2: Josef Bican\'s goals per match ratio is 0.64, indicating his exceptional goal-scoring ability.Most important fact 3: Josef Bican\'s career spanned from 1928 to 1955, during which he played for various clubs and national teams.Additional key fact 1: Josef Bican\'s record of 805 goals has stood the test of time, and he remains one of the most iconic and revered footballers in history.Additional key fact 2: Josef Bican\'s achievements and records in world foo

  5%|▌         | 1/20 [01:52<35:39, 112.62s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.17632868885993958, 'start': 396, 'end': 407, 'answer': 'Josef Bican'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.07008715718984604, 'start': 2, 'end': 13, 'answer': 'Josef Bican'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 3.505080530885607e-05, 'start': 2, 'end': 13, 'answer': 'Josef Bican'}
{'rougeLsum': 23.981900452488688, 'length': 310.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Final ans: ['Here\'s a comprehensive cheat sheet for the query:**Most Important Facts:**1. The original artist of "The Sound of Silence" is Simon & Garfunkel, an American music duo consisting of Paul Simon and Art Garfunkel.2. The song was written by Paul Simon over several mon

 10%|█         | 2/20 [03:41<33:12, 110.67s/it]

...

 95%|█████████▌| 19/20 [45:27<02:10, 130.65s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.1720561385154724, 'start': 212, 'end': 224, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.2675960063934326, 'start': 294, 'end': 304, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.16925226151943207, 'start': 212, 'end': 224, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.12236358225345612, 'start': 294, 'end': 304, 'answer': 'Ethan Peck'}
{'rougeLsum': 24.68354430379747, 'length': 238.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
Final ans: ['("Microsoft Liv

100%|██████████| 20/20 [48:40<00:00, 146.00s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.5430126190185547, 'start': 53, 'end': 75, 'answer': 'video editing software'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.8486161231994629, 'start': 53, 'end': 66, 'answer': 'video editing'}
{'rougeLsum': 11.165048543689318, 'length': 364.0, 'str_em': 50.0, 'Disambig-F1': 40.0}


rougeLsum       20.540544
length         312.200000
str_em          64.166667
Disambig-F1     49.952381
dtype: float64

In [34]:
scores_df.to_csv('./results/self-refine-02-09_results.csv', index=False)